# CIEL-0 GPU Training — Google Colab (free tier, no payment)

This picks up where the CPU-based training left off, now with:
- **Free GPU acceleration** (Runtime → Change runtime type → T4 GPU, if not already set)
- **Google Drive persistence** — Colab sessions disconnect and wipe local files after a few hours of inactivity; checkpoints saved to Drive survive that.
- **A bigger model** than was practical on CPU — more parameters, bigger context window, since GPU makes this tractable.

Run cells top to bottom. Re-running this notebook later automatically resumes from your last saved checkpoint in Drive.

In [ ]:
# Step 1: Mount Google Drive — this is what makes training progress SURVIVE
# across Colab sessions. Without this, everything is lost when the runtime
# disconnects (which happens automatically after ~90 min idle or ~12hrs total
# on the free tier).
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/ciel0'
os.makedirs(PROJECT_DIR, exist_ok=True)
print(f'Checkpoints and data will persist in: {PROJECT_DIR}')

In [ ]:
# Step 2: Confirm GPU is actually active. If this prints False, go to
# Runtime -> Change runtime type -> Hardware accelerator -> GPU (T4),
# then Runtime -> Restart session, and re-run from Step 1.
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

In [ ]:
# Step 3: Get training data. Starting with the same Shakespeare corpus for
# continuity with your CPU-trained checkpoints — swap this out for a bigger,
# more diverse corpus once you're ready to tackle Gap 3 (training data
# diversity). Cached in Drive so it only downloads once.
data_path = os.path.join(PROJECT_DIR, 'tinyshakespeare.txt')
if not os.path.exists(data_path):
    import urllib.request
    url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
    urllib.request.urlretrieve(url, data_path)
    print('Downloaded.')
else:
    print('Already cached in Drive.')

with open(data_path, 'r', encoding='utf-8') as f:
    text = f.read()
print(f'Corpus size: {len(text):,} characters')

In [ ]:
# Step 4: Model definition — same architecture family as your CPU version,
# but sized up now that GPU makes it practical. Compare to your CPU runs:
#   CPU: d_model=128, n_heads=4, n_layers=4, block_size=64-256  (~940K params)
#   GPU: d_model=256, n_heads=8, n_layers=6, block_size=512     (much bigger)
import torch.nn as nn
import torch.nn.functional as F

chars = sorted(set(text))
vocab_size = len(chars)
char_to_id = {ch: i for i, ch in enumerate(chars)}
id_to_char = {i: ch for i, ch in enumerate(chars)}
def encode(s): return [char_to_id[c] for c in s]
def decode(ids): return ''.join(id_to_char[i] for i in ids)

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

def get_batch(data, block_size, batch_size):
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix]).to(device)
    y = torch.stack([data[i+1:i+block_size+1] for i in ix]).to(device)
    return x, y

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.n_heads = n_heads; self.d_head = d_model // n_heads
        self.W_q = nn.Linear(d_model, d_model); self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model); self.W_out = nn.Linear(d_model, d_model)
    def forward(self, x):
        b, t, d = x.shape
        Q = self.W_q(x).view(b,t,self.n_heads,self.d_head).transpose(1,2)
        K = self.W_k(x).view(b,t,self.n_heads,self.d_head).transpose(1,2)
        V = self.W_v(x).view(b,t,self.n_heads,self.d_head).transpose(1,2)
        out = F.scaled_dot_product_attention(Q, K, V, is_causal=True)
        out = out.transpose(1,2).contiguous().view(b,t,d)
        return self.W_out(out)

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model,d_ff), nn.GELU(), nn.Linear(d_ff,d_model))
    def forward(self, x): return self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, n_heads)
        self.feedforward = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model); self.norm2 = nn.LayerNorm(d_model)
    def forward(self, x):
        x = x + self.attention(self.norm1(x))
        x = x + self.feedforward(self.norm2(x))
        return x

class TinyGPT(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers, d_ff, block_size):
        super().__init__()
        self.block_size = block_size
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(block_size, d_model)
        self.blocks = nn.ModuleList([TransformerBlock(d_model,n_heads,d_ff) for _ in range(n_layers)])
        self.final_norm = nn.LayerNorm(d_model)
        self.output_head = nn.Linear(d_model, vocab_size)
    def forward(self, idx, targets=None):
        b, t = idx.shape
        x = self.token_embedding(idx) + self.position_embedding(torch.arange(t, device=idx.device))
        for block in self.blocks: x = block(x)
        x = self.final_norm(x)
        logits = self.output_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss
    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=0.8):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)
        return idx

# Scaled-up config — practical now that a GPU is doing the work.
D_MODEL, N_HEADS, N_LAYERS, D_FF, BLOCK_SIZE = 256, 8, 6, 1024, 512
model = TinyGPT(vocab_size, D_MODEL, N_HEADS, N_LAYERS, D_FF, BLOCK_SIZE).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {n_params:,}  (compare to ~940K on CPU)')

In [ ]:
# Step 5: Train, with checkpointing to Drive (survives session disconnects)
# and mixed precision (torch.autocast) — this is the piece that DIDN'T help
# on CPU (see 13_training_infrastructure.py) but genuinely speeds things up
# and saves memory on a real GPU with tensor cores.
CKPT_PATH = os.path.join(PROJECT_DIR, 'ciel0_gpu_checkpoint.pt')
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

start_step = 0
if os.path.exists(CKPT_PATH):
    ckpt = torch.load(CKPT_PATH, weights_only=True)
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    start_step = ckpt['total_steps']
    print(f'Resumed from step {start_step}')
else:
    print('Starting fresh.')

STEPS_THIS_RUN = 3000  # adjust based on how much time you have this session
batch_size = 64
scaler = torch.amp.GradScaler('cuda', enabled=(device=='cuda'))

import time
start = time.time()
for step in range(STEPS_THIS_RUN):
    xb, yb = get_batch(train_data, BLOCK_SIZE, batch_size)
    with torch.autocast(device_type=device, dtype=torch.bfloat16, enabled=(device=='cuda')):
        logits, loss = model(xb, yb)
    optimizer.zero_grad()
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    if step % 200 == 0 or step == STEPS_THIS_RUN - 1:
        elapsed = time.time() - start
        print(f'step {start_step+step} | loss={loss.item():.4f} | {elapsed:.0f}s ({elapsed/(step+1):.3f}s/step)')

torch.save({
    'model_state': model.state_dict(), 'optimizer_state': optimizer.state_dict(),
    'total_steps': start_step + STEPS_THIS_RUN, 'vocab': chars,
    'd_model': D_MODEL, 'n_heads': N_HEADS, 'n_layers': N_LAYERS, 'd_ff': D_FF, 'block_size': BLOCK_SIZE,
}, CKPT_PATH)
print(f'Saved to Drive at step {start_step + STEPS_THIS_RUN}. Safe to disconnect — re-run this notebook later to resume.')

In [ ]:
# Step 6: Generate a sample to see current quality
model.eval()
idx = torch.tensor([encode('\nROMEO:')], dtype=torch.long).to(device)
out = model.generate(idx, max_new_tokens=300, temperature=0.8)
print(decode(out[0].tolist()))
model.train()